## Tylko testy do większych zbiorów danych

In [1]:
import os
import pandas as pd
from torchvision.io import read_image
import re
import wfdb
import wfdb.processing
import scipy
from torch.utils.data import Dataset
import numpy as np
import json
import torch.nn as nn
import torch
from tqdm import tqdm
import torch.nn.functional as F

In [1]:
import os
import re
import numpy as np
import torch
from torch.utils.data import Dataset
import wfdb
import scipy.signal

def extract_qrs_segment(signal, qrs_index, N):
    """
    Ekstrahuje segment sygnału EKG wokół zadanego indeksu QRS.
    Dopełnia brakujące próbki medianą sygnału, jeśli QRS jest blisko granic.
    """
    start_idx = qrs_index - N
    end_idx = qrs_index + N + 1  # +1, bo Python używa wykluczającego zakresu
    
    if start_idx < 0:
        padding_left = np.median(signal[:end_idx])
        segment = np.concatenate([np.full(-start_idx, padding_left), signal[:end_idx]])
    elif end_idx > len(signal):
        padding_right = np.median(signal[start_idx:])
        segment = np.concatenate([signal[start_idx:], np.full(end_idx - len(signal), padding_right)])
    else:
        segment = signal[start_idx:end_idx]
    
    return segment

class SHDB_QRS(Dataset):
    def __init__(self, N, dataset_dir, fs=100):
        """
        N - liczba próbek przed i po załamku QRS w segmencie.
        dataset_dir - ścieżka do katalogu z danymi EKG.
        fs - docelowa częstotliwość próbkowania.
        """
        self.N = N
        self.fs = fs
        self.qrs_segments = []
        self.labels = []

        exclusion_lst = []  # Lista plików do wykluczenia, jeśli potrzebna
        for file in os.listdir(dataset_dir):
            name = re.match(r'^(.*\d\d+)\.atr$', file)
            if name and name.group(1) not in exclusion_lst:
                print(f"Przetwarzanie: {name.group(1)}")
                record = wfdb.rdsamp(f"{dataset_dir}{name.group(1)}")
                annotation = wfdb.rdann(f"{dataset_dir}{name.group(1)}", 'atr')
                signal = record[0][:, 0]
                fs_original = record[1]["fs"]

                # Filtracja sygnału
                cutOff = 20
                b, a = scipy.signal.butter(5, cutOff, fs=fs_original, btype='low', analog=False)
                filtered_signal = scipy.signal.lfilter(b, a, signal)

                # Resampling do docelowej częstotliwości próbkowania
                num_samples_target = int(filtered_signal.shape[0] * fs / fs_original)
                resampled_signal = scipy.signal.resample(filtered_signal, num_samples_target)
                resampled_annotations = (annotation.sample * fs) / fs_original
                resampled_annotations = resampled_annotations.astype(int)

                # Detekcja QRS (np. użycie XQRS lub anotacji)
                xqrs = wfdb.processing.XQRS(sig=resampled_signal, fs=fs)
                xqrs.detect()  # Wykrywanie QRS
                qrs_indices = xqrs.qrs_inds

                # Tworzenie segmentów QRS
                for qrs_idx in qrs_indices:
                    # Znajdź najbliższą etykietę w anotacjach
                    nearest_label_idx = np.argmin(np.abs(resampled_annotations - qrs_idx))
                    label = annotation.aux_note[nearest_label_idx]
                    
                    # Sprawdź, czy to AFIB, czy normalny rytm
                    is_afib = label == '(AFIB'
                    self.qrs_segments.append(extract_qrs_segment(resampled_signal, qrs_idx, N))
                    self.labels.append(1 if is_afib else 0)

    def __len__(self):
        return len(self.qrs_segments)

    def __getitem__(self, idx):
        segment = torch.Tensor(self.qrs_segments[idx]).unsqueeze(0)  # Dodaj wymiar kanału
        label = self.labels[idx]
        return segment, label

ModuleNotFoundError: No module named 'torch._C'

In [7]:
ds = SHDB_QRS(50,'/home/zosia/dnn/dnn_ecg/physionet.org/shdb-af-a-japanese-holter-ecg-database-of-atrial-fibrillation-1.0.0/',fs=100)

Przetwarzanie: 030
Learning initial signal parameters...
Found 8 beats during learning. Initializing using learned parameters
Running QRS detection...
QRS detection complete.
Przetwarzanie: 086
Learning initial signal parameters...
Found 8 beats during learning. Initializing using learned parameters
Running QRS detection...
QRS detection complete.
Przetwarzanie: 042
Learning initial signal parameters...
Found 8 beats during learning. Initializing using learned parameters
Running QRS detection...
QRS detection complete.
Przetwarzanie: 047
Learning initial signal parameters...
Found 8 beats during learning. Initializing using learned parameters
Running QRS detection...
QRS detection complete.
Przetwarzanie: 040
Learning initial signal parameters...
Found 8 beats during learning. Initializing using learned parameters
Running QRS detection...
QRS detection complete.
Przetwarzanie: 035
Learning initial signal parameters...
Found 8 beats during learning. Initializing using learned parameters

In [8]:
from torch.utils.data import DataLoader, random_split
train_set, val_set = random_split(ds, [0.8, 0.2])
train = DataLoader(train_set, batch_size=32, shuffle=True)
val = DataLoader(val_set, batch_size=32, shuffle=True)

## Model a La resnet

In [18]:
class ResNetBlock(nn.Module):
    def __init__(self,in_channels, out_channels):
        """
        output same as input
        """
        super(ResNetBlock, self).__init__()
        self.conv1 = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
                        nn.BatchNorm1d(out_channels),
                        nn.ReLU(inplace=False))  # Changed inplace to False
        self.conv2 = nn.Sequential(
                        nn.Conv1d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
                        nn.BatchNorm1d(out_channels),
                        nn.ReLU(inplace=False))
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        if(in_channels != out_channels):
            self.residual = nn.Sequential(
                nn.Conv1d(self.in_channels, out_channels, kernel_size=1, stride=1),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self,x):
        out = self.conv1(x)
        out = self.conv2(out)
        if self.in_channels != self.out_channels:
            residual = self.residual(x)
        else:
            residual = x
        return F.relu(out + residual, inplace=False)



class ResNetLike(nn.Module):
    def __init__(self, input = 100, input_ch = 1, num_classes = 2):
        super(ResNetLike, self).__init__()
        self.model = nn.Sequential(
            nn.Conv1d(input_ch, 64, kernel_size=7, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            ResNetBlock(64,64),
            ResNetBlock(64,64),
            ResNetBlock(64,64),
            ResNetBlock(64,128),    # out 1 x 128 x n
            nn.MaxPool1d(2),        # out 1 x 128 x n//2
            ResNetBlock(128,128),
            ResNetBlock(128,128),
            ResNetBlock(128,256),
            nn.MaxPool1d(2),        # out 1 x 256 x n//2
            ResNetBlock(256,256),
            ResNetBlock(256,256),
            ResNetBlock(256,512),
            nn.MaxPool1d(2),        # out 1 x 512 x n//8
            nn.Flatten(),
            nn.Linear(512*(input//8), 256),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
        self.model.to('cuda:0')

    def forward(self, x):

        return self.model(x)
    
    def train_model(self, train_loader, valid_loader, num_epochs = 5, learning_rate=0.001, save_best = False, save_thr = 0.94):
        best_accuracy = 0.0
        total_step = len(train_loader)
        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.RMSprop(self.parameters(), lr=learning_rate, weight_decay = 0.005, momentum = 0.9)  

        for epoch in range(num_epochs):
            # self.train()
            correct = 0
            total = 0
            for i, (images, labels) in enumerate(tqdm(train_loader)):
                # Move tensors to the configured device
                images = images.float().to("cuda")
                labels = labels.type(torch.LongTensor)
                labels = labels.to("cuda")


                optimizer.zero_grad()

                # Forward pass
                outputs = self.forward(images)
                loss = criterion(outputs, labels)
                # Backward and optimize
                loss.backward()
                
                optimizer.step()

                # accuracy
                _, predicted = torch.max(outputs.data, 1)
                correct += (torch.eq(predicted, labels)).sum().item()
                total += labels.size(0)

                del images, labels, outputs

            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}, Accuracy: {:.4f}'
                            .format(epoch+1, num_epochs, i+1, total_step, loss.item(), (float(correct))/total))


            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Validation
            with torch.no_grad():
                correct = 0
                total = 0
                for images, labels in valid_loader:
                    images = images.float().to("cuda")
                    labels = labels.to("cuda")
                    outputs = self.forward(images)
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (torch.eq(predicted, labels)).sum().item()
                    del images, labels, outputs
                if(((100 * correct / total) > best_accuracy) and save_best and ((100 * correct / total) > save_thr)):
                    torch.save(self.state_dict(), "costamtest-DVS2.pt")

                print('Accuracy of the network: {} %'.format( 100 * correct / total))

In [19]:
model_res = ResNetLike()


In [20]:
model_res.train_model(train,val,num_epochs=5, save_best=True)

100%|██████████| 275489/275489 [1:20:29<00:00, 57.04it/s]


Epoch [1/2], Step [275489/275489], Loss: 0.0008, Accuracy: 0.9996
Accuracy of the network: 99.99087984122757 %


100%|██████████| 275489/275489 [1:19:18<00:00, 57.89it/s]


Epoch [2/2], Step [275489/275489], Loss: 0.0008, Accuracy: 0.9996
Accuracy of the network: 99.99087984122757 %


```
Epoch [1/90], Step [41922/41922], Loss: 0.2852, Accuracy: 0.9416
Accuracy of the network: 94.53356551193593 %
  2%|▏         | 876/41922 [00:14<11:42, 58.41it/s]

Epoch [1/90], Step [41922/41922], Loss: 0.2852, Accuracy: 0.9416
Accuracy of the network: 94.53356551193593 %
  2%|▏         | 876/41922 [00:14<11:42, 58.41it/s]

```

In [22]:
torch.save(model_res.state_dict(), "best_resnet50_MINST-DVS22.pt")

In [ ]:
model_res.train_model(train,val,num_epochs=90,learning_rate=0.0001,save_best=True)